# 🎓 Chapter 21: Correlation vs Causation
*Track 1 — Students | Tier 2*

> **🎬 Hook:** Ice cream sales and drowning rates are strongly correlated (r = 0.85).
> Should we ban ice cream to save lives? Understanding why this is absurd is the whole lesson.

**🎯 Objectives:** Compute Pearson r, interpret it correctly, recognize confounders and spurious correlations.

## 📖 Section 1 — Concept Review

### Pearson Correlation Coefficient
$$r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \sum(y_i-\bar{y})^2}}$$

Properties: -1 ≤ r ≤ 1, r=0 (uncorrelated), r=±1 (perfect linear)

### Correlation ≠ Causation
Four reasons for spurious correlation:
1. **Common cause (confounder):** Z causes both X and Y (summer → ice cream AND drowning)
2. **Reverse causation:** Maybe Y causes X, not X causing Y
3. **Coincidence:** Random chance, especially with many comparisons
4. **Selection bias:** The data wasn't collected randomly

### When CAN you infer causation?
- **Randomized Controlled Trial (RCT)**: Gold standard
- **Natural experiment**: Random-ish assignment in the real world
- **Instrumental variables**: More advanced causal methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")
np.random.seed(42)

# ── Visualize different correlations ──
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

scenarios = [
    ('Strong positive
r ≈ 0.95', 0.95),
    ('Moderate positive
r ≈ 0.5', 0.5),
    ('No correlation
r ≈ 0', 0.0),
    ('Moderate negative
r ≈ -0.5', -0.5),
    ('Non-linear
(r misleads!)', None),
    ('Outlier effect', 0.9),
]

n = 100
for ax, (title, r) in zip(axes.flatten(), scenarios):
    if title == 'Non-linear
(r misleads!)':
        x = np.linspace(0, 2*np.pi, n)
        y = np.sin(x) + np.random.normal(0, 0.2, n)
        corr = np.corrcoef(x, y)[0,1]
    elif title == 'Outlier effect':
        x = np.random.normal(0, 1, n-5)
        y = x * 0.1 + np.random.normal(0, 1, n-5)
        x = np.append(x, [5, 6, 7, 8, 9])
        y = np.append(y, [5, 6, 7, 8, 9])
        corr = np.corrcoef(x, y)[0,1]
    else:
        cov = [[1, r], [r, 1]]
        data = np.random.multivariate_normal([0,0], cov, n)
        x, y = data[:,0], data[:,1]
        corr = np.corrcoef(x, y)[0,1]

    ax.scatter(x, y, alpha=0.6, s=25, color='#3498db')
    m, b, _, _, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m*x_line + b, 'r-', lw=2)
    ax.set_title(f'{title}
Actual r = {corr:.2f}', fontweight='bold', fontsize=9)

plt.suptitle("Correlation Patterns: r can be misleading!", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch21_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Spurious Correlation: The Confounder ──
np.random.seed(42)
n = 52  # 52 weeks of data

# Temperature is the hidden confounder (summer heat)
temperature = np.tile(np.sin(np.linspace(0, 2*np.pi, 52)), 1) * 20 + 25
ice_cream_sales = temperature * 50 + np.random.normal(0, 200, n)
drowning_rate   = temperature * 2  + np.random.normal(0, 5, n)

r_spurious = np.corrcoef(ice_cream_sales, drowning_rate)[0,1]
r_partial   = np.corrcoef(ice_cream_sales - 50*temperature,
                           drowning_rate - 2*temperature)[0,1]

print(f"🍦 Spurious Correlation Example")
print(f"r(ice cream, drowning) = {r_spurious:.3f}  ← Strong!")
print(f"r after removing temperature effect = {r_partial:.3f}  ← Near zero!")
print()
print("The confounder (temperature) created the illusion of causation.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ice_cream_sales, drowning_rate, c=temperature, cmap='Reds', alpha=0.8, s=40)
axes[0].set_xlabel('Ice Cream Sales'); axes[0].set_ylabel('Drowning Rate')
axes[0].set_title(f'🍦 Spurious: r={r_spurious:.2f}
(both caused by temperature!)', fontweight='bold')

weeks = np.arange(52)
ax2 = axes[1]
ax2.plot(weeks, ice_cream_sales/ice_cream_sales.max(), label='Ice Cream (normalized)', color='#e74c3c')
ax2.plot(weeks, drowning_rate/drowning_rate.max(), label='Drowning Rate (normalized)', color='#3498db')
ax2.plot(weeks, temperature/temperature.max(), label='Temperature (the confounder)', color='#f39c12', linestyle='--')
ax2.set_xlabel('Week of Year'); ax2.set_ylabel('Normalized Value')
ax2.set_title('🌡️ Temperature drives both variables', fontweight='bold')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.savefig('ch21_spurious.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**P1:** Calculate r for: x=[1,2,3,4,5], y=[2,4,5,4,5].

**P2:** Identify the likely confounder: "Cities with more hospitals have higher death rates."

**P3:** Is each causal or merely correlational?
  a) Smoking and lung cancer (RCT would be unethical, but mechanism is known)
  b) Shoe size and reading ability in children
  c) Exercise and lower blood pressure (RCT data available)

<details><summary>💡 Solutions</summary>

**P1:** r ≈ 0.90 (positive, strong)

**P2:** Population size — larger cities have more hospitals AND more deaths.

**P3:** a) Causal (mechanism + natural experiments), b) Confounder: age (older kids = bigger feet + better reading), c) Causal (RCT evidence)
</details>

In [ ]:
x = np.array([1,2,3,4,5]); y = np.array([2,4,5,4,5])
r, p = stats.pearsonr(x, y)
print(f"P1: r = {r:.4f}, p = {p:.4f}")

# Anscombe's Quartet: same r, very different data
print("
Anscombe's Quartet: all have r ≈ 0.816, but look different!")
anscombe = sns.load_dataset('anscombe')
for dataset in ['I','II','III','IV']:
    d = anscombe[anscombe.dataset==dataset]
    r_val = np.corrcoef(d.x, d.y)[0,1]
    print(f"  Dataset {dataset}: r = {r_val:.3f}")

## 🎯 Recap
1. Correlation measures LINEAR association (−1 to +1).
2. r ≠ causation: confounders, reverse causation, and coincidence all create spurious correlations.
3. Only randomized experiments allow causal claims.

**Next:** [Chapter 22 — Exam Strategy & Common Mistakes]